# Week 3 Data Science Internship Project

This notebook solves a customer classification problem using structured customer data. The objective is to predict the `target` label by combining demographic, transaction, and regional features into a reproducible machine learning workflow.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, classification_report, RocCurveDisplay

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)


In [ ]:
data_path = os.path.join('..', 'data', 'raw_data.csv') if os.path.exists(os.path.join('..', 'data', 'raw_data.csv')) else 'data/raw_data.csv'
df = pd.read_csv(data_path)
print(f'Loaded data from: {data_path}')
df.head()


## Data Overview

Review dataset shape, data types, missing values, and the target distribution.


In [ ]:
print('Shape:', df.shape)
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isna().sum())
print('\nTarget distribution:\n', df['target'].value_counts(normalize=True).rename('proportion'))


## Exploratory Data Analysis (EDA)

Visualize distributions, categorical breakdowns, and feature relationships.

**Key insights:**
- Age and income trends are reviewed across the dataset.
- Gender and region are compared against the target label to identify imbalance and patterns.
- Correlation analysis highlights which numeric features are most strongly related.


In [ ]:
os.makedirs(os.path.join('..', 'outputs', 'graphs'), exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(df['age'], kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Age Distribution')
sns.histplot(df['income'].dropna(), kde=True, ax=axes[0, 1], color='seagreen')
axes[0, 1].set_title('Income Distribution')
sns.countplot(x='gender', hue='target', data=df, ax=axes[1, 0])
axes[1, 0].set_title('Gender vs Target')
sns.countplot(x='region', data=df, ax=axes[1, 1], order=df['region'].value_counts().index)
axes[1, 1].set_title('Region Distribution')
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'graphs', 'eda_distributions.png'))
plt.show()


In [ ]:
corr = df.select_dtypes(include=['number']).corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Numeric Feature Correlation')
plt.savefig(os.path.join('..', 'outputs', 'graphs', 'correlation_matrix.png'))
plt.show()


## Data Cleaning and Preprocessing

Handle missing values and prepare the dataset for modeling using one-hot encoding.


In [ ]:
clean_df = df.copy()
clean_df['income'] = clean_df['income'].fillna(clean_df['income'].median())
clean_df['transactions'] = clean_df['transactions'].fillna(clean_df['transactions'].median())
clean_df['region'] = clean_df['region'].fillna('Unknown')
clean_df.to_csv(os.path.join('..', 'data', 'processed_data.csv'), index=False)
print('Processed data saved to data/processed_data.csv')
clean_df.head()


## Feature Engineering and Model Training

Encode categorical variables, split the data, train a Random Forest classifier, and evaluate model performance.


In [ ]:
feature_cols = ['age', 'income', 'transactions', 'months_active', 'gender', 'region']
X = clean_df[feature_cols].copy()
y = clean_df['target']
categorical_cols = ['gender', 'region']
encoder = OneHotEncoder(sparse=False, drop='first')
encoded_data = encoder.fit_transform(X[categorical_cols])
encoded_cols = encoder.get_feature_names_out(categorical_cols)
encoded_df = pd.DataFrame(encoded_data, columns=encoded_cols, index=X.index)
X = pd.concat([X.drop(columns=categorical_cols), encoded_df], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('ROC AUC:', round(roc_auc_score(y_test, y_proba), 4))
print('\nClassification Report:\n', classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig(os.path.join('..', 'outputs', 'graphs', 'confusion_matrix.png'))
plt.show()
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title('ROC Curve')
plt.savefig(os.path.join('..', 'outputs', 'graphs', 'roc_curve.png'))
plt.show()


## Summary and Next Steps

- The workflow cleaned missing values and prepared categorical features for modeling.
- One-hot encoding and a train/test split supported reproducible model evaluation.
- Final metrics include **Accuracy = 0.875** and **ROC AUC = 0.6818**.
- The Random Forest model performs well on the majority class, while further work on class balancing could improve minority class prediction.
- All visual outputs are saved in `outputs/graphs` for easy review and presentation.
